# F2_04 Integración

## Finalidad

Construir la **big table analítica** (una fila por diputado y por votación) que integra las
cuatro tablas procesadas en F2_03:

- `F2/data/processed/detalle_votaciones_procesado.csv` — tabla base/vertebral.
- `F2/data/processed/proyecto_ley_procesado.csv`
- `F2/data/processed/diputados_procesados.csv`
- `F2/data/processed/militancias_analiticas.csv`

**Granularidad:** una fila = un voto emitido por un diputado en una votación. Clave esperada:
`["votacion_id", "diputado_id"]`. `detalle_votaciones_procesado` es la columna vertebral: la
cantidad final de filas debe ser exactamente igual a su cantidad inicial de filas.

**Salida:** `F2/data/processed/big_table_analitica.csv`.

**Criterio rector:** este notebook **no toma nuevas decisiones analíticas**. Todas las decisiones
sobre militancias y calidad de datos ya vienen resueltas desde F2_03 (incluida la extensión de la
militancia de Bugueño hasta el fin del período, con el mismo criterio ya aplicado a Veloso). Aquí
solo se integra, se valida y se documenta.


## 1. Configuración

In [1]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Fecha de ejecución UTC:", datetime.now(timezone.utc).isoformat())


Fecha de ejecución UTC: 2026-09-15T00:05:08.812368+00:00


In [2]:
# Detectar la carpeta F2 de forma robusta.

cwd = Path.cwd().resolve()

candidatos = [cwd] + list(cwd.parents)

BASE_DIR = None

for candidato in candidatos:
    if candidato.name == "F2" and (candidato / "data").exists():
        BASE_DIR = candidato
        break

if BASE_DIR is None:
    for candidato in candidatos:
        posible = candidato / "F2"
        if (posible / "data").exists():
            BASE_DIR = posible
            break

if BASE_DIR is None:
    raise FileNotFoundError(
        "No se pudo localizar la carpeta F2/data. "
        "Ejecuta el notebook desde el repositorio del proyecto."
    )

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVO_SALIDA = "big_table_analitica.csv"
ARCHIVO_DIAGNOSTICO = "diagnostico_integracion.csv"

print(f"BASE_DIR: {BASE_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")


BASE_DIR: /Users/ygallardo/Documents/Magister UNAB/Programacion para la cienia de datos/grupo_1_programacion_ciencia_datos/F2
PROCESSED_DIR: /Users/ygallardo/Documents/Magister UNAB/Programacion para la cienia de datos/grupo_1_programacion_ciencia_datos/F2/data/processed


## 2. Carga de datos

Se cargan las cuatro tablas de entrada, con los identificadores forzados a texto (para que pandas
no los transforme en números) y las fechas como `datetime`. Se crea además un identificador
temporal de fila (`fila_voto_id`), que permite comprobar que ningún voto se pierde ni se duplica
durante los cruces; se elimina antes de exportar (sección 11).


In [3]:
archivos_entrada = {
    "detalle_votaciones_procesado.csv": ["diputado_id", "votacion_id"],
    "proyecto_ley_procesado.csv": ["Id"],
    "diputados_procesados.csv": ["diputado_id", "periodo_id"],
    "militancias_analiticas.csv": ["diputado_id", "partido_id"],
}

for nombre_archivo in archivos_entrada:
    ruta = PROCESSED_DIR / nombre_archivo
    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontró {ruta}. Ejecuta F2_03_procesamiento_validacion.ipynb primero."
        )

df_detalle = pd.read_csv(
    PROCESSED_DIR / "detalle_votaciones_procesado.csv",
    dtype={"diputado_id": str, "votacion_id": str},
)
df_detalle["fecha"] = pd.to_datetime(df_detalle["fecha"], errors="raise")

df_proyecto = pd.read_csv(
    PROCESSED_DIR / "proyecto_ley_procesado.csv",
    dtype={"Id": str},
)
df_proyecto["Fecha"] = pd.to_datetime(df_proyecto["Fecha"], errors="raise")

df_diputados = pd.read_csv(
    PROCESSED_DIR / "diputados_procesados.csv",
    dtype={"diputado_id": str, "periodo_id": str},
)

df_militancias = pd.read_csv(
    PROCESSED_DIR / "militancias_analiticas.csv",
    dtype={"diputado_id": str, "partido_id": str},
)
for columna in ["fecha_inicio", "fecha_termino", "fecha_termino_original"]:
    df_militancias[columna] = pd.to_datetime(df_militancias[columna], errors="raise")

# Identificador temporal de fila: permite comprobar que ningún voto se pierde ni se duplica.
df_detalle["fila_voto_id"] = df_detalle["votacion_id"] + "_" + df_detalle["diputado_id"]

for nombre, df in [
    ("detalle_votaciones_procesado", df_detalle),
    ("proyecto_ley_procesado", df_proyecto),
    ("diputados_procesados", df_diputados),
    ("militancias_analiticas", df_militancias),
]:
    print(f"{nombre}: {df.shape[0]} filas x {df.shape[1]} columnas")

display(df_detalle.head())


detalle_votaciones_procesado: 1993 filas x 20 columnas
proyecto_ley_procesado: 15 filas x 15 columnas
diputados_procesados: 157 filas x 11 columnas
militancias_analiticas: 399 filas x 10 columnas


,diputado_id,nombre,apellido_paterno,apellido_materno,opcion_codigo,opcion_voto,votacion_id,descripcion,fecha,total_si,total_no,total_abstencion,total_dispensado,quorum_codigo,quorum,resultado_codigo,resultado,tipo_codigo,tipo,fila_voto_id
0,803,René,Alinco,Bustos,0,En Contra,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_803
1,815,Sergio,Bobadilla,Muñoz,2,Abstención,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_815
2,872,Jaime,Mulet,Martínez,1,Afirmativo,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_872
3,917,Gastón,Von Mühlenbrock,Zamora,0,En Contra,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_917
4,948,Gaspar,Rivas,Sánchez,2,Abstención,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_948


## 3. Validaciones previas

Antes de integrar, se comprueba la cobertura referencial entre las cuatro tablas mediante
anti-uniones. Estas validaciones confirman que existe *alguna* fila candidata para cada voto en
`proyecto_ley`, `diputados` y `militancias_analiticas` — no que la militancia cubra la fecha
exacta del voto, que es lo que valida la sección 7.


In [4]:
controles_referenciales = []


def anti_join(izquierda, derecha, clave_izquierda, clave_derecha, nombre_control):
    if (
        izquierda is None
        or derecha is None
        or clave_izquierda not in izquierda.columns
        or clave_derecha not in derecha.columns
    ):
        controles_referenciales.append(
            {"validacion": nombre_control, "fuera_catalogo": None, "estado": "ADVERTENCIA"}
        )
        return

    catalogo = derecha[[clave_derecha]].drop_duplicates()

    comparacion = izquierda.merge(
        catalogo, how="left", left_on=clave_izquierda, right_on=clave_derecha, indicator=True
    )

    fuera = comparacion[comparacion["_merge"] == "left_only"]

    controles_referenciales.append(
        {
            "validacion": nombre_control,
            "fuera_catalogo": len(fuera),
            "estado": "OK" if len(fuera) == 0 else "ERROR",
        }
    )


anti_join(df_detalle, df_proyecto, "votacion_id", "Id", "Votos con votación inexistente en proyecto_ley")
anti_join(df_proyecto, df_detalle, "Id", "votacion_id", "Votaciones de proyecto_ley sin detalle")
anti_join(df_detalle, df_diputados, "diputado_id", "diputado_id", "Votos con diputado inexistente")
anti_join(
    df_detalle,
    df_militancias,
    "diputado_id",
    "diputado_id",
    "Votos de diputados sin ninguna militancia analítica registrada",
)

controles_referenciales_df = pd.DataFrame(controles_referenciales)
display(controles_referenciales_df)

errores_previos = controles_referenciales_df[controles_referenciales_df["estado"] == "ERROR"]
if not errores_previos.empty:
    display(errores_previos)
    raise ValueError(
        f"{len(errores_previos)} validaciones previas fallaron; revisar F2_03 antes de integrar."
    )


,validacion,fuera_catalogo,estado
0,Votos con votación inexistente en proyecto_ley,0,OK
1,Votaciones de proyecto_ley sin detalle,0,OK
2,Votos con diputado inexistente,0,OK
3,Votos de diputados sin ninguna militancia anal...,0,OK


## 4. Orden de integración

| Paso | Base | Se une con | Clave | Tipo | Cardinalidad esperada |
|---|---|---|---|---|---|
| 1 | `detalle_votaciones_procesado` (1993 filas) | `proyecto_ley_procesado` | `votacion_id` = `Id` | `left` | muchos a uno |
| 2 | resultado del paso 1 | `diputados_procesados` | `diputado_id` = `diputado_id` | `left` | muchos a uno |
| 3 | resultado del paso 2 | `militancias_analiticas` | `diputado_id` + vigencia en la fecha del voto | join temporal | un registro vigente por voto |

```
detalle_votaciones_procesado (1993 filas = 1 voto por fila)
   fila_voto_id = votacion_id + "_" + diputado_id
        │
        ├── LEFT JOIN proyecto_ley_procesado, por votacion_id = Id (muchos a uno)
        │
        ├── LEFT JOIN diputados_procesados, por diputado_id (muchos a uno)
        │
        └── join temporal con militancias_analiticas,
            por diputado_id y vigencia en la fecha del voto
                │
                ▼
        big_table_analitica
```


## 5. Integración con proyecto_ley

`proyecto_ley_procesado` comparte columnas semánticamente redundantes con `detalle` bajo otro
nombre (`Descripcion/Fecha/TotalSi/TotalNo/TotalAbstencion/TotalDispensado/Quorum/Resultado/Tipo`
en PascalCase, frente a `descripcion/fecha/total_si/total_no/total_abstencion/total_dispensado/
quorum/resultado/tipo` en snake_case en `detalle`). No colisionan por nombre, pero representan el
mismo dato: se usan aquí solo para verificar consistencia y luego se descartan — al `big_table`
solo se incorporan las columnas realmente nuevas (`numero_boletin`,
`tipo_votacion_proyecto_ley`, `articulo`, `tramite_constitucional`, `tramite_reglamentario`), para
no dejar sobrevivir dos versiones del mismo dato bajo distinto nombre.

La comparación de texto (`Descripcion`/`descripcion`) elimina espacios antes de comparar: ambas
tablas provienen de dos llamadas independientes a la API y difieren de forma sistemática en
espacios (p. ej. `"N°11092-07"` vs `"N° 11092-07"`) sin que el contenido cambie realmente.


In [5]:
mapeo_columnas = {
    "Descripcion": "descripcion",
    "Fecha": "fecha",
    "TotalSi": "total_si",
    "TotalNo": "total_no",
    "TotalAbstencion": "total_abstencion",
    "TotalDispensado": "total_dispensado",
    "Quorum": "quorum",
    "Resultado": "resultado",
    "Tipo": "tipo",
}

metadata_detalle = (
    df_detalle.groupby("votacion_id")[list(mapeo_columnas.values())].first().reset_index()
)

comparacion_proyecto = df_proyecto.merge(
    metadata_detalle, left_on="Id", right_on="votacion_id", how="inner"
)

# Normaliza espacios antes de comparar texto: proyecto_ley y detalle provienen de dos llamadas
# independientes a la API y difieren de forma sistemática en espacios (p. ej. "N°11092-07" vs
# "N° 11092-07" en la Descripcion de las 15 votaciones) sin que el contenido cambie. Se eliminan
# todos los espacios en vez de solo colapsarlos, porque la diferencia es de espacios ausentes,
# no de espacios repetidos.
def normalizar_texto_comparacion(serie):
    return serie.astype(str).str.replace(r"\s+", "", regex=True)


inconsistencias_proyecto = []
for col_proyecto, col_detalle in mapeo_columnas.items():
    diferentes = (
        normalizar_texto_comparacion(comparacion_proyecto[col_proyecto])
        != normalizar_texto_comparacion(comparacion_proyecto[col_detalle])
    )
    if diferentes.any():
        inconsistencias_proyecto.append(
            {
                "campo_proyecto": col_proyecto,
                "campo_detalle": col_detalle,
                "votaciones_inconsistentes": int(diferentes.sum()),
            }
        )

inconsistencias_proyecto_df = pd.DataFrame(inconsistencias_proyecto)
print(f"Inconsistencias detalle vs. proyecto_ley: {len(inconsistencias_proyecto)}")
if inconsistencias_proyecto:
    display(inconsistencias_proyecto_df)

columnas_proyecto_nuevas = [
    "Id",
    "numero_boletin",
    "TipoVotacionProyectoLey",
    "Articulo",
    "TramiteConstitucional",
    "TramiteReglamentario",
]

big_table = df_detalle.merge(
    df_proyecto[columnas_proyecto_nuevas],
    how="left",
    left_on="votacion_id",
    right_on="Id",
    validate="many_to_one",
)
big_table = big_table.drop(columns=["Id"]).rename(
    columns={
        "TipoVotacionProyectoLey": "tipo_votacion_proyecto_ley",
        "Articulo": "articulo",
        "TramiteConstitucional": "tramite_constitucional",
        "TramiteReglamentario": "tramite_reglamentario",
    }
)

assert len(big_table) == len(df_detalle), "El join con proyecto_ley alteró la cantidad de filas."
print(f"big_table tras integrar proyecto_ley: {big_table.shape[0]} filas x {big_table.shape[1]} columnas")


Inconsistencias detalle vs. proyecto_ley: 0
big_table tras integrar proyecto_ley: 1993 filas x 25 columnas


## 6. Integración con diputados

`nombre`, `apellido_paterno` y `apellido_materno` existen con el mismo nombre en `detalle` y en
`diputados_procesados`. Antes de unir se verifica que coincidan por `diputado_id`; al confirmarse
idénticas, el cruce **no vuelve a traerlas** — solo se incorporan las columnas nuevas de
`diputados_procesados`, evitando duplicados y sufijos `_x`/`_y`.


In [6]:
columnas_identidad = ["nombre", "apellido_paterno", "apellido_materno"]

verificacion_identidad = big_table[["diputado_id"] + columnas_identidad].merge(
    df_diputados[["diputado_id"] + columnas_identidad],
    on="diputado_id",
    how="left",
    suffixes=("_detalle", "_maestra"),
)

diferencia_identidad = pd.Series(False, index=verificacion_identidad.index)
for columna in columnas_identidad:
    diferencia_identidad |= (
        verificacion_identidad[f"{columna}_detalle"] != verificacion_identidad[f"{columna}_maestra"]
    )

n_diferencias_identidad = int(diferencia_identidad.sum())
print(f"Diferencias de identidad detalle vs. diputados_procesados: {n_diferencias_identidad}")

if n_diferencias_identidad > 0:
    display(verificacion_identidad[diferencia_identidad])
    raise ValueError(
        "Se encontraron diferencias de identidad entre detalle y diputados_procesados. "
        "Esto indica un problema en F2_03, no algo que F2_04 deba corregir."
    )

columnas_diputados_nuevas = [
    "diputado_id",
    "fecha_nacimiento",
    "sexo_valor",
    "sexo_desc",
    "periodo_id",
    "fecha_inicio_periodo",
    "fecha_termino_periodo",
    "tiene_votos",
]

big_table = big_table.merge(
    df_diputados[columnas_diputados_nuevas],
    how="left",
    on="diputado_id",
    validate="many_to_one",
)

assert len(big_table) == len(df_detalle), "El join con diputados alteró la cantidad de filas."
assert not big_table.columns.duplicated().any(), "Hay columnas duplicadas tras el join con diputados."
print(f"big_table tras integrar diputados: {big_table.shape[0]} filas x {big_table.shape[1]} columnas")


Diferencias de identidad detalle vs. diputados_procesados: 0
big_table tras integrar diputados: 1993 filas x 32 columnas


## 7. Integración temporal con militancias

Paso crítico: se asigna a cada voto la militancia vigente en `militancias_analiticas` según un
intervalo **cerrado**:

$$ fecha\_inicio \leq fecha\_votacion \leq fecha\_termino $$

Si `fecha_termino` es nula, el intervalo se considera abierto:

$$ fecha\_inicio \leq fecha\_votacion $$

(Nota: esto es deliberadamente distinto del intervalo semiabierto `fecha_inicio \leq fecha <
fecha_termino` usado en el EDA de F2_02 — aquí se sigue el criterio cerrado especificado para esta
integración.)

El notebook se detiene si, tras el filtro, algún voto queda con cero militancias aplicables, más
de una militancia aplicable, o con partido analítico vacío.


In [7]:
# 7a. Re-verificar ausencia de solapamientos en militancias_analiticas (no asumir que F2_03 quedó bien).
solapamientos_militancias = []

for diputado_id, grupo in df_militancias.groupby("diputado_id"):
    grupo = grupo.sort_values("fecha_inicio").reset_index(drop=True)
    for i in range(len(grupo) - 1):
        termino_a = grupo.loc[i, "fecha_termino"]
        termino_a = termino_a if pd.notna(termino_a) else pd.Timestamp.max
        inicio_b = grupo.loc[i + 1, "fecha_inicio"]
        if inicio_b <= termino_a:
            solapamientos_militancias.append(
                {
                    "diputado_id": diputado_id,
                    "partido_a": grupo.loc[i, "partido_id"],
                    "partido_b": grupo.loc[i + 1, "partido_id"],
                }
            )

if solapamientos_militancias:
    display(pd.DataFrame(solapamientos_militancias))
    raise ValueError(
        f"militancias_analiticas tiene {len(solapamientos_militancias)} intervalos solapados; "
        "corregir F2_03 antes de continuar."
    )

print("Sin solapamientos en militancias_analiticas.")


Sin solapamientos en militancias_analiticas.


In [8]:
# 7b. Asignar, para cada voto, la militancia vigente en su fecha.
militancias_por_diputado = {
    diputado_id: grupo.sort_values("fecha_inicio").reset_index(drop=True)
    for diputado_id, grupo in df_militancias.groupby("diputado_id")
}

diagnostico_militancia = []
resultados_militancia = []

for _, voto in big_table[["fila_voto_id", "diputado_id", "fecha"]].iterrows():
    grupo = militancias_por_diputado.get(voto["diputado_id"])
    fecha = voto["fecha"]

    if grupo is None:
        vigentes = grupo
        cantidad = 0
    else:
        vigentes = grupo[
            (grupo["fecha_inicio"] <= fecha)
            & (grupo["fecha_termino"].isna() | (fecha <= grupo["fecha_termino"]))
        ]
        cantidad = len(vigentes)

    estado = "OK" if cantidad == 1 else ("SIN_MILITANCIA" if cantidad == 0 else "AMBIGUA")

    diagnostico_militancia.append(
        {
            "fila_voto_id": voto["fila_voto_id"],
            "diputado_id": voto["diputado_id"],
            "fecha": fecha,
            "militancias_vigentes": cantidad,
            "estado": estado,
        }
    )

    fila_militancia = vigentes.iloc[0] if cantidad == 1 else None
    resultados_militancia.append(
        {
            "fila_voto_id": voto["fila_voto_id"],
            "partido_id": fila_militancia["partido_id"] if fila_militancia is not None else None,
            "partido_nombre": fila_militancia["partido_nombre"] if fila_militancia is not None else None,
            "partido_alias": fila_militancia["partido_alias"] if fila_militancia is not None else None,
        }
    )

diagnostico_militancia_df = pd.DataFrame(diagnostico_militancia)
resultados_militancia_df = pd.DataFrame(resultados_militancia)

casos_invalidos = diagnostico_militancia_df[diagnostico_militancia_df["estado"] != "OK"]
if not casos_invalidos.empty:
    display(casos_invalidos)
    raise ValueError(
        f"{len(casos_invalidos)} votos no tienen exactamente 1 militancia vigente en su fecha "
        "(ver tabla anterior). Revisar militancias_analiticas.csv (F2_03) antes de continuar."
    )

big_table = big_table.merge(resultados_militancia_df, on="fila_voto_id", how="left", validate="one_to_one")

if big_table[["partido_id", "partido_nombre", "partido_alias"]].isna().any().any():
    raise ValueError("Se encontraron filas con partido analítico vacío tras el join temporal.")

print(
    "Integración temporal con militancias: OK, exactamente 1 militancia vigente por cada uno de "
    f"{len(big_table)} votos."
)


Integración temporal con militancias: OK, exactamente 1 militancia vigente por cada uno de 1993 votos.


## 8. Validación de casos especiales

Comprobaciones puntuales sobre las decisiones ya tomadas en F2_03 — validaciones, no nuevas
transformaciones.


In [9]:
def verificar_partido_unico_esperado(diputado_id, alias_esperado, nombre_caso):
    alias_obtenidos = sorted(
        big_table.loc[big_table["diputado_id"] == diputado_id, "partido_alias"].unique()
    )
    ok = alias_obtenidos == [alias_esperado]
    print(f"{nombre_caso} ({diputado_id}): alias obtenidos={alias_obtenidos}, esperado=[{alias_esperado}] -> {'OK' if ok else 'ERROR'}")
    if not ok:
        raise ValueError(f"Caso especial {nombre_caso} no coincide con lo esperado tras la integración.")


verificar_partido_unico_esperado("1017", "UDI", "Carter")
verificar_partido_unico_esperado("1114", "FRVS", "Bugueño")
verificar_partido_unico_esperado("1180", "RD", "Veloso")


Carter (1017): alias obtenidos=['UDI'], esperado=[UDI] -> OK
Bugueño (1114): alias obtenidos=['FRVS'], esperado=[FRVS] -> OK
Veloso (1180): alias obtenidos=['RD'], esperado=[RD] -> OK


## 9. Validaciones de la big table

La integración se considera válida solo si se cumplen simultáneamente las siguientes condiciones.
Se reutiliza el mismo patrón de matriz de validaciones que F2_02.

Dos columnas quedan fuera de las comprobaciones de completitud (6 y 7) porque su vacío es una
característica ya documentada de la fuente, no una falla de esta integración: `articulo` (nulo
para la votación 20627, ya señalado en F2_02/F2_03) y `fecha_termino_periodo` (nula para los 157
diputados porque el período 10 sigue vigente). Ambas quedan igualmente disponibles en la columna
correspondiente de `big_table`; solo no se exige que estén presentes para aprobar la integración.


In [10]:
matriz_validaciones = []


def agregar_validacion(nombre, resultado, estado, detalle=""):
    matriz_validaciones.append(
        {"validacion": nombre, "resultado": resultado, "estado": estado, "detalle": detalle}
    )


# 1. Conteo de filas igual al detalle original.
agregar_validacion(
    "Conteo de filas = detalle original",
    f"{len(big_table)} vs {len(df_detalle)}",
    "OK" if len(big_table) == len(df_detalle) else "ERROR",
)

# 2. votacion_id + diputado_id continúa siendo clave única.
n_dup_clave = int(big_table.duplicated(subset=["votacion_id", "diputado_id"], keep=False).sum())
agregar_validacion("Clave votacion_id + diputado_id única", n_dup_clave, "OK" if n_dup_clave == 0 else "ERROR")

# 3. fila_voto_id continúa siendo única.
n_dup_fila = int(big_table["fila_voto_id"].duplicated(keep=False).sum())
agregar_validacion("fila_voto_id única", n_dup_fila, "OK" if n_dup_fila == 0 else "ERROR")

# 4. Sin columnas duplicadas.
n_col_dup = int(big_table.columns.duplicated().sum())
agregar_validacion("Sin columnas duplicadas", n_col_dup, "OK" if n_col_dup == 0 else "ERROR")

# 5. Sin sufijos accidentales _x / _y.
sufijos = [c for c in big_table.columns if c.endswith("_x") or c.endswith("_y")]
agregar_validacion("Sin columnas con sufijo _x/_y", sufijos, "OK" if not sufijos else "ERROR")

# 6. Todos los votos tienen datos del proyecto de ley.
# "articulo" queda fuera de esta comprobación: proyecto_ley_procesado trae 1 valor nulo en
# Articulo (votación 20627), ya documentado como hallazgo de calidad en F2_02/F2_03 -- no es
# una falla de este join, es una característica ya conocida de la fuente.
cols_proyecto = [
    "numero_boletin", "tipo_votacion_proyecto_ley",
    "tramite_constitucional", "tramite_reglamentario",
]
n_sin_proyecto = int(big_table[cols_proyecto].isna().any(axis=1).sum())
agregar_validacion("Todos los votos tienen datos de proyecto_ley", n_sin_proyecto, "OK" if n_sin_proyecto == 0 else "ERROR")

n_sin_articulo = int(big_table["articulo"].isna().sum())
print(
    f"Nota: {n_sin_articulo} votos sin 'articulo' (votación con Articulo nulo en proyecto_ley_procesado, "
    "hallazgo ya documentado en F2_02/F2_03; no se cuenta como falla de integración)."
)

# 7. Todos los votos tienen datos del diputado.
# "fecha_termino_periodo" queda fuera de esta comprobación: es nula para los 157 diputados
# (período 10 aún vigente, sin fecha de término), no indica que el join haya fallado.
cols_diputado = [
    "fecha_nacimiento", "sexo_valor", "sexo_desc", "periodo_id",
    "fecha_inicio_periodo", "tiene_votos",
]
n_sin_diputado = int(big_table[cols_diputado].isna().any(axis=1).sum())
agregar_validacion("Todos los votos tienen datos de diputado", n_sin_diputado, "OK" if n_sin_diputado == 0 else "ERROR")

# 8. Todos los votos tienen partido analítico.
n_sin_partido = int(big_table[["partido_id", "partido_nombre", "partido_alias"]].isna().any(axis=1).sum())
agregar_validacion("Todos los votos tienen partido analítico", n_sin_partido, "OK" if n_sin_partido == 0 else "ERROR")

# 9. La militancia asignada estaba vigente en la fecha del voto (reusa el diagnóstico de la sección 7).
n_militancia_no_ok = int((diagnostico_militancia_df["estado"] != "OK").sum())
agregar_validacion(
    "Militancia vigente en la fecha del voto (exactamente 1)",
    n_militancia_no_ok,
    "OK" if n_militancia_no_ok == 0 else "ERROR",
)

# 10. Los totales agregados por votación son consistentes con el catálogo (reusa la sección 5).
agregar_validacion(
    "Totales/metadatos de detalle vs. proyecto_ley coinciden",
    len(inconsistencias_proyecto),
    "OK" if not inconsistencias_proyecto else "ERROR",
)

# 11. Las columnas originales del detalle permanecen en la salida.
columnas_detalle_originales = [c for c in df_detalle.columns if c != "fila_voto_id"]
faltantes = sorted(set(columnas_detalle_originales) - set(big_table.columns))
agregar_validacion("Columnas originales de detalle preservadas", faltantes, "OK" if not faltantes else "ERROR")

# 12. No se introdujeron nuevas categorías de voto (guardia de regresión).
categorias_nuevas = set()
for columna in ["opcion_voto", "opcion_codigo"]:
    antes = set(df_detalle[columna].dropna().unique())
    despues = set(big_table[columna].dropna().unique())
    categorias_nuevas |= despues - antes
agregar_validacion(
    "Sin nuevas categorías de opcion_voto/opcion_codigo",
    sorted(categorias_nuevas),
    "OK" if not categorias_nuevas else "ERROR",
)

matriz_validaciones_df = pd.DataFrame(matriz_validaciones)
display(matriz_validaciones_df)

errores_integracion = matriz_validaciones_df[matriz_validaciones_df["estado"] != "OK"]
if not errores_integracion.empty:
    display(errores_integracion)
    raise ValueError(f"{len(errores_integracion)} validaciones de la big table fallaron; ver tabla anterior.")


Nota: 122 votos sin 'articulo' (votación con Articulo nulo en proyecto_ley_procesado, hallazgo ya documentado en F2_02/F2_03; no se cuenta como falla de integración).


,validacion,resultado,estado,detalle
0,Conteo de filas = detalle original,1993 vs 1993,OK,
1,Clave votacion_id + diputado_id única,0,OK,
2,fila_voto_id única,0,OK,
3,Sin columnas duplicadas,0,OK,
4,Sin columnas con sufijo _x/_y,[],OK,
5,Todos los votos tienen datos de proyecto_ley,0,OK,
6,Todos los votos tienen datos de diputado,0,OK,
7,Todos los votos tienen partido analítico,0,OK,
8,Militancia vigente en la fecha del voto (exact...,0,OK,
9,Totales/metadatos de detalle vs. proyecto_ley ...,0,OK,


## 10. Diagnóstico final

In [11]:
diagnostico_integracion = [
    {"metrica": "filas totales", "valor": len(big_table)},
    {"metrica": "votaciones únicas", "valor": int(big_table["votacion_id"].nunique())},
    {"metrica": "diputados únicos", "valor": int(big_table["diputado_id"].nunique())},
    {"metrica": "partidos analíticos únicos (partido_alias)", "valor": int(big_table["partido_alias"].nunique())},
    {"metrica": "rango de fechas", "valor": f"{big_table['fecha'].min()} → {big_table['fecha'].max()}"},
    {"metrica": "duplicados de clave (votacion_id + diputado_id)", "valor": n_dup_clave},
    {"metrica": "votos sin datos de diputado", "valor": n_sin_diputado},
    {"metrica": "votos sin datos de proyecto_ley", "valor": n_sin_proyecto},
    {"metrica": "votos sin militancia vigente", "valor": int((diagnostico_militancia_df["estado"] == "SIN_MILITANCIA").sum())},
    {"metrica": "votos con más de una militancia vigente", "valor": int((diagnostico_militancia_df["estado"] == "AMBIGUA").sum())},
    {"metrica": "columnas originales (detalle_votaciones_procesado)", "valor": len(columnas_detalle_originales)},
    {"metrica": "columnas agregadas por proyecto_ley", "valor": len(columnas_proyecto_nuevas) - 1},
    {"metrica": "columnas agregadas por diputados_procesados", "valor": len(columnas_diputados_nuevas) - 1},
    {"metrica": "columnas agregadas por militancias_analiticas", "valor": 3},
    {"metrica": "columnas totales en big_table", "valor": big_table.shape[1]},
]

diagnostico_integracion_df = pd.DataFrame(diagnostico_integracion)
display(diagnostico_integracion_df)


,metrica,valor
0,filas totales,1993
1,votaciones únicas,15
2,diputados únicos,151
3,partidos analíticos únicos (partido_alias),23
4,rango de fechas,2023-05-08 19:05:22 → 2024-08-26 19:03:25
5,duplicados de clave (votacion_id + diputado_id),0
6,votos sin datos de diputado,0
7,votos sin datos de proyecto_ley,0
8,votos sin militancia vigente,0
9,votos con más de una militancia vigente,0


## 11. Exportación

Se ordena por fecha, votación y diputado; se elimina el identificador temporal `fila_voto_id`; se
restablece el índice. Las fechas se mantienen en el formato `YYYY-MM-DD HH:MM:SS` que ya usan
todos los CSV de `F2/data/processed/` (es el formato por defecto de pandas al exportar columnas
`datetime`, ISO 8601 con espacio en vez de `T`).


In [12]:
big_table_exportar = (
    big_table
    .drop(columns=["fila_voto_id"])
    .sort_values(["fecha", "votacion_id", "diputado_id"])
    .reset_index(drop=True)
)

ruta_salida = PROCESSED_DIR / ARCHIVO_SALIDA
big_table_exportar.to_csv(ruta_salida, index=False, encoding="utf-8")
print(f"Guardado: {ruta_salida} ({big_table_exportar.shape[0]} filas x {big_table_exportar.shape[1]} columnas)")

ruta_diagnostico = PROCESSED_DIR / ARCHIVO_DIAGNOSTICO
diagnostico_integracion_df.to_csv(ruta_diagnostico, index=False, encoding="utf-8")
print(f"Guardado: {ruta_diagnostico} ({len(diagnostico_integracion_df)} filas)")


Guardado: /Users/ygallardo/Documents/Magister UNAB/Programacion para la cienia de datos/grupo_1_programacion_ciencia_datos/F2/data/processed/big_table_analitica.csv (1993 filas x 34 columnas)
Guardado: /Users/ygallardo/Documents/Magister UNAB/Programacion para la cienia de datos/grupo_1_programacion_ciencia_datos/F2/data/processed/diagnostico_integracion.csv (15 filas)


## 12. Criterio de término

El veredicto se calcula a partir del resultado real de la sección 9 (no se escribe a mano), para
que nunca quede una celda "Integración aprobada" visualmente presente si una ejecución posterior
llegara a fallar.


In [13]:
todo_ok = matriz_validaciones_df["estado"].eq("OK").all()

if todo_ok:
    display(Markdown(
        "## ✅ Integración aprobada\n\n"
        "Todas las validaciones de la Sección 9 pasaron. "
        f"`{ARCHIVO_SALIDA}` refleja el resultado final."
    ))
else:
    display(Markdown(
        "## ❌ Integración rechazada\n\n"
        "Hay validaciones con estado distinto de OK; ver la matriz de la Sección 9."
    ))
    raise RuntimeError("Integración rechazada: revisar matriz_validaciones_df.")


## ✅ Integración aprobada

Todas las validaciones de la Sección 9 pasaron. `big_table_analitica.csv` refleja el resultado final.

In [14]:
big_table.head()

,diputado_id,nombre,apellido_paterno,apellido_materno,opcion_codigo,opcion_voto,votacion_id,descripcion,fecha,total_si,total_no,total_abstencion,total_dispensado,quorum_codigo,quorum,resultado_codigo,resultado,tipo_codigo,tipo,fila_voto_id,numero_boletin,tipo_votacion_proyecto_ley,articulo,tramite_constitucional,tramite_reglamentario,fecha_nacimiento,sexo_valor,sexo_desc,periodo_id,fecha_inicio_periodo,fecha_termino_periodo,tiene_votos,partido_id,partido_nombre,partido_alias
0,803,René,Alinco,Bustos,0,En Contra,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_803,11092-07,Única,Proposición de la Comisión Mixta recaída en el...,Comisión Mixta,Sin Informe,1958-06-02,1,Masculino,10,2026-03-10 23:59:59,NaN,True,IND,Independientes,IND
1,815,Sergio,Bobadilla,Muñoz,2,Abstención,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_815,11092-07,Única,Proposición de la Comisión Mixta recaída en el...,Comisión Mixta,Sin Informe,1958-03-25,1,Masculino,10,2026-03-10 23:59:59,NaN,True,UDI,Unión Demócrata Independiente,UDI
2,872,Jaime,Mulet,Martínez,1,Afirmativo,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_872,11092-07,Única,Proposición de la Comisión Mixta recaída en el...,Comisión Mixta,Sin Informe,1963-08-03,1,Masculino,10,2026-03-10 23:59:59,NaN,True,FRVS,Federación Regionalista Verde Social,FRVS
3,917,Gastón,Von Mühlenbrock,Zamora,0,En Contra,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_917,11092-07,Única,Proposición de la Comisión Mixta recaída en el...,Comisión Mixta,Sin Informe,1954-12-26,1,Masculino,10,2026-03-10 23:59:59,NaN,True,UDI,Unión Demócrata Independiente,UDI
4,948,Gaspar,Rivas,Sánchez,2,Abstención,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_948,11092-07,Única,Proposición de la Comisión Mixta recaída en el...,Comisión Mixta,Sin Informe,1978-05-17,1,Masculino,10,2026-03-10 23:59:59,NaN,True,PDG,Partido de la Gente,PDG
